# Physics-Guided Graph Learning for Wind-Turbine Root-Cause Localization

This version runs **only on the CARE-to-Compare Kaggle dataset**. It treats each event CSV as its own turbine/event sequence, uses `event_info.csv` as the fault/event authority, and uses `feature_description.csv` as the sensor authority. No synthetic data or injected faults are used.

The loader deliberately does not concatenate events, discard turbines, cap the number of files, or identify faults from filenames.


In [ ]:

import os, math, json, glob, random, warnings, itertools
from dotenv import load_dotenv
load_dotenv()
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import networkx as nx
from scipy import stats

from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
sns.set_theme(style="whitegrid", font_scale=0.9)


## 0. Environment Setup

### Local VS Code setup (no Colab / no Drive mounting)

This notebook runs locally in VS Code. Put the extracted CARE-to-Compare dataset under `data/care_data/` in the project root. No Kaggle token is needed if the dataset is already downloaded. If you need the dataset, download it using Kaggle CLI on your own machine, then extract it into that folder.


**Dataset required:** `data/care_data/` must contain one or more farm folders with `event_info.csv`, `feature_description.csv`, and the event CSV files. Keep the original directory structure.


In [ ]:
CONFIG = dict(
    # Mount this Kaggle dataset with "Add Input". For local use, set this to its extracted root.
    DATA_DIR = os.getenv("CARE_DATA_DIR", "data/care_data"),
    TARGET_FARM = None,       # e.g. "Wind Farm A"; None selects a documented anomaly event
    TARGET_EVENT_ID = None,   # exact event_id from event_info.csv; None selects deterministically
    MAX_VARIABLES = 20,
    WINDOW_SIZE = 24,
    TRAIN_RATIO = 0.8,        # only partitions the dataset-provided train segment into train/validation
    VAL_RATIO = 0.2,
    CORR_THRESH = 0.70,
    MI_THRESH = 0.70,
    CAUSAL_THRESH = 0.55,
    CAUSAL_WINDOW = 240,
    CAUSAL_STEP = 120,
    CAUSAL_MAXLAG = 3,
    HIDDEN_DIM = 32, EMB_DIM = 16, GC_DEPTH = 2, BETA = 0.6,
    NUM_TC_GC_LAYERS = 2, GUMBEL_TAU = 0.5,
    BATCH_SIZE = 64, EPOCHS = 12, PATIENCE = 4, LR = 1e-3, WEIGHT_DECAY = 1e-5,
    ANOMALY_THRESH = 0.5, MC_DROPOUT_PASSES = 8,
    MI_TOP_K = 3,
    CAUSAL_TOP_K = 2,
    CAUSAL_MIN_EFFECT = 0.0001
)
print(json.dumps(CONFIG, indent=2))


## 1. CARE-to-Compare data contract

The Kaggle dataset is a comma-separated repackaging of CARE-to-Compare (the original release is semicolon-separated). It contains 95 event-centred CSV sequences across 36 turbines and three wind farms. Each sequence has `time_stamp`, `id`, `asset_id`, `status_type_id`, `train_test`, and sensor-statistic columns. `event_info.csv` is the authoritative registry: its `event_id` matches the CSV stem; `event_label` is `normal`/`anomaly`; its start/end timestamps and IDs define the event window; `event_description` contains a root-cause description when disclosed. In newer releases it also has `asset_id`.

`feature_description.csv` is the authoritative sensor dictionary: `sensor_name`, `statistic_type`, `description`, `unit`, `is_angle`, and `is_counter`. The data-quality notes recommend average (`Avg`/`average`) channels for analysis: Min/Max/Std may be implausible. The loader therefore reads **all** event CSVs into a catalog, selects one documented target sequence without joining unrelated turbines/events, and uses every eligible average numeric sensor in a diagnostic-value selection step. Unknown sensors remain eligible.


In [ ]:
# 1.1 Explicit CARE schema loader — no filename or keyword fault heuristics
from pathlib import Path

CARE_META_FILES = {"event_info.csv", "feature_description.csv"}
META_COLUMNS = {"time_stamp", "id", "asset_id", "status_type_id", "train_test"}

def read_care_csv(path, **kwargs):
    """Read either Kaggle's comma export or the original semicolon CARE release."""
    for sep in (",", ";"):
        try:
            frame = pd.read_csv(path, sep=sep, **kwargs)
            if frame.shape[1] > 1:
                frame.columns = [str(c).strip() for c in frame.columns]
                return frame
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    raise ValueError(f"Could not parse CARE CSV: {path}")

def find_care_farms(data_dir):
    root = Path(data_dir)
    if not root.exists():
        raise FileNotFoundError(
            f"CARE dataset not found at {root}. In Kaggle use Add Input; locally set CONFIG['DATA_DIR'] to the extracted dataset root."
        )
    event_info_paths = sorted(root.rglob("event_info.csv"))
    if not event_info_paths:
        raise FileNotFoundError("No event_info.csv found. This notebook requires the CARE-to-Compare dataset layout.")
    farms = []
    for info_path in event_info_paths:
        farm_dir = info_path.parent
        feature_path = farm_dir / "feature_description.csv"
        if not feature_path.exists():
            raise FileNotFoundError(f"Missing feature_description.csv beside {info_path}")
        events = read_care_csv(info_path)
        required = {"event_id", "event_label", "event_start", "event_end", "event_start_id", "event_end_id"}
        missing = required - set(events.columns)
        if missing:
            raise ValueError(f"{info_path} is not a CARE event registry; missing {sorted(missing)}")
        features = read_care_csv(feature_path)
        required_features = {"sensor_name", "description", "unit", "is_angle", "is_counter"}
        missing = required_features - set(features.columns)
        if missing:
            raise ValueError(f"{feature_path} is not a CARE sensor dictionary; missing {sorted(missing)}")
        csv_by_stem = {p.stem: p for p in farm_dir.rglob("*.csv") if p.name.lower() not in CARE_META_FILES}
        event_rows = events.copy()
        event_rows["event_id"] = event_rows["event_id"].astype(str)
        event_rows["event_csv"] = event_rows["event_id"].map(csv_by_stem)
        if event_rows["event_csv"].isna().any():
            absent = event_rows.loc[event_rows.event_csv.isna(), "event_id"].tolist()
            raise FileNotFoundError(f"Registry events missing CSV files in {farm_dir}: {absent[:10]}")
        farms.append(dict(name=farm_dir.name, directory=farm_dir, events=event_rows, features=features))
    return farms

def build_event_catalog(farms):
    rows = []
    for farm in farms:
        for _, event in farm["events"].iterrows():
            rows.append({
                "farm": farm["name"], "event_id": str(event.event_id), "event_label": str(event.event_label).lower(),
                "asset_id": event.get("asset_id", pd.NA), "event_start": event.event_start, "event_end": event.event_end,
                "event_start_id": event.event_start_id, "event_end_id": event.event_end_id,
                "event_description": event.get("event_description", event.get("description", pd.NA)),
                "event_csv": str(event.event_csv),
            })
    return pd.DataFrame(rows).sort_values(["farm", "event_id"]).reset_index(drop=True)

def select_target_event(catalog):
    choices = catalog.copy()
    if CONFIG["TARGET_FARM"] is not None:
        choices = choices[choices.farm.eq(CONFIG["TARGET_FARM"])]
    if CONFIG["TARGET_EVENT_ID"] is not None:
        choices = choices[choices.event_id.eq(str(CONFIG["TARGET_EVENT_ID"]))]
    if choices.empty:
        raise ValueError("TARGET_FARM/TARGET_EVENT_ID does not identify an event in event_info.csv")
    if CONFIG["TARGET_EVENT_ID"] is None:
        # Prefer an anomaly whose disclosed event_description supports a root-cause case study; deterministic tie-break.
        desc = choices.event_description.fillna("").astype(str).str.strip()
        choices = choices.assign(_documented=desc.ne(""))
        choices = choices.sort_values(["event_label", "_documented", "farm", "event_id"],
                                      ascending=[True, False, True, True])
    return choices.iloc[0].drop(labels=["_documented"], errors="ignore")

def sensor_metadata(feature_description, event_columns):
    fd = feature_description.copy()
    fd["sensor_name"] = fd.sensor_name.astype(str).str.strip()
    if "statistic_type" not in fd.columns:
      fd["statistic_type"] = "avg"
    else:
      fd["statistic_type"] = fd["statistic_type"].astype(str).str.lower().str.strip()
    # Use only the documented 10-minute average channels; retain anonymized/unknown sensors.
    fd = fd[fd.statistic_type.isin({"avg", "average"})].copy()
    # CARE releases differ: the dictionary may say sensor_12 + Avg while the data column is sensor_12_avg.
    def resolve_data_column(row):
        base = row.sensor_name
        for candidate in (base, f"{base}_{row.statistic_type}", f"{base}_avg", f"{base}_average"):
            if candidate in event_columns:
                return candidate
        return None
    fd["data_column"] = fd.apply(resolve_data_column, axis=1)
    fd = fd.dropna(subset=["data_column"]).copy()
    fd["description"] = fd.description.fillna("").astype(str)
    patterns = {
      "wind": r"\bwind\s*speed\b|\bwindspeed\b",
      "power": r"\b(active|reactive)\s+power\b|\bpower\b",
      "speed": r"\b(rpm|rotor\s+speed|generator\s+speed|generator\s+rpm)\b",
      "angle": r"\b(pitch|angle|direction|azimuth|yaw)\b",
      "temperature": r"\btemp(?:erature)?\b",
    }

    text = fd["description"].fillna("").str.lower()

    tag_matches = pd.DataFrame({
        category: text.str.contains(pattern, regex=True)
        for category, pattern in patterns.items()
    })

    fd["semantic_tags"] = tag_matches.apply(
        lambda row: [category for category, matched in row.items() if matched],
        axis=1,
    )

    fd["category"] = fd["semantic_tags"].apply(
        lambda tags: tags[0] if len(tags) == 1 else ("mixed" if len(tags) > 1 else "unknown")
    )

    fd.loc[
        fd["category"].eq("unknown")
        & fd["is_angle"].astype(str).str.lower().eq("true"),
        "category",
    ] = "angle"

    return fd.drop_duplicates("data_column")

def diagnostic_feature_selection(frame, metadata, max_vars):
    """Quality + relevance + redundancy selection; no name filtering or variance ranking."""
    candidates = []
    for _, m in metadata.iterrows():
        col = m.data_column
        x = pd.to_numeric(frame[col], errors="coerce")
        valid = x.notna().mean()
        iqr = x.quantile(.75) - x.quantile(.25)
        if valid >= .70 and np.isfinite(iqr) and iqr > 0:
            candidates.append((col, valid, float(iqr)))
    if len(candidates) < 6:
        raise ValueError(f"Only {len(candidates)} usable average sensor channels; need at least 6.")
    cols = [x[0] for x in candidates]
    sample = frame[cols].apply(pd.to_numeric, errors="coerce").interpolate(limit_direction="both").ffill().bfill()
    sample = sample.iloc[np.linspace(0, len(sample)-1, min(3000, len(sample)), dtype=int)]
    # Relevance is median mutual information with documented operational anchors; otherwise temporal persistence.
    anchors = metadata.loc[metadata.category.isin(["wind", "power", "speed"]), "data_column"].tolist()
    anchors = [c for c in anchors if c in cols][:5]
    relevance = {}
    for col in cols:
        if anchors:
            vals = [mutual_info_regression(sample[[a]], sample[col], random_state=SEED)[0] for a in anchors if a != col]
            relevance[col] = float(np.median(vals)) if vals else 0.0
        else:
            relevance[col] = abs(float(sample[col].autocorr(lag=1)))
    quality = dict((c, v) for c, v, _ in candidates)
    ranked = sorted(cols, key=lambda c: (relevance[c], quality[c]), reverse=True)
    selected = []
    for col in ranked:
        if all(abs(sample[col].corr(sample[chosen])) < .98 for chosen in selected):
            selected.append(col)
        if len(selected) == max_vars:
            break
    meta = metadata.set_index("data_column").loc[selected].reset_index()
    return meta

def load_care_target(data_dir):
    farms = find_care_farms(data_dir)
    catalog = build_event_catalog(farms)  # all 95 event/turbine files are represented; none are silently omitted
    target = select_target_event(catalog)
    farm = next(f for f in farms if f["name"] == target.farm)
    raw = read_care_csv(target.event_csv)
    required = META_COLUMNS - {"asset_id"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Event {target.event_id} missing required CARE columns: {sorted(missing)}")
    raw["time_stamp"] = pd.to_datetime(raw.time_stamp, errors="coerce")
    raw = raw.dropna(subset=["time_stamp"]).sort_values(["time_stamp", "id"]).reset_index(drop=True)
    meta = sensor_metadata(farm["features"], raw.columns)
    selected = diagnostic_feature_selection(raw, meta, CONFIG["MAX_VARIABLES"])
    symbol_map = {f"S{i+1:02d}": {"column": r.data_column, "sensor_name": r.sensor_name, "category": r.category,
                                   "description": r.description, "unit": r.unit}
                  for i, (_, r) in enumerate(selected.iterrows())}
    return raw, symbol_map, target, catalog


In [ ]:
# 1.2 Actual-data-only execution
# Change TARGET_FARM and TARGET_EVENT_ID in CONFIG to select any catalog event.
# There is intentionally no synthetic fallback: a missing or malformed data mount is an explicit error.


In [ ]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nDataset folder exists:")
print(Path("data/care_data").exists())

print("\nEvent metadata exists:")
print(Path("data/care_data/event_info.csv").exists())

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

# If the notebook is running from the project root:
CONFIG["DATA_DIR"] = str(PROJECT_DIR / "data" / "care_data")

print("Dataset path:", CONFIG["DATA_DIR"])
print("Path exists:", Path(CONFIG["DATA_DIR"]).exists())

In [ ]:

from pathlib import Path

PROJECT_DIR = Path.cwd().parent

CONFIG["DATA_DIR"] = str(PROJECT_DIR / "data" / "care_data")

print("Dataset path:", CONFIG["DATA_DIR"])
print("Dataset folder exists:", Path(CONFIG["DATA_DIR"]).exists())
print("Event metadata exists:", (Path(CONFIG["DATA_DIR"]) / "event_info.csv").exists())


In [ ]:
raw_event, symbol_map, active_event, EVENT_CATALOG = load_care_target(CONFIG["DATA_DIR"])
feature_columns = [v["column"] for v in symbol_map.values()]
data_df = raw_event[feature_columns].apply(pd.to_numeric, errors="coerce")
data_df.columns = list(symbol_map.keys())
sequence_meta = raw_event[[c for c in META_COLUMNS if c in raw_event.columns]].copy()

print(f"CARE catalog: {len(EVENT_CATALOG)} event CSVs | {EVENT_CATALOG.farm.nunique()} farms | "
      f"{EVENT_CATALOG.asset_id.nunique(dropna=True)} recorded assets")
print("Selected event:", active_event[["farm", "event_id", "event_label", "asset_id", "event_description"]].to_dict())
print(f"Selected sequence: {len(data_df):,} timestamps × {len(feature_columns)} average sensor channels")
display(EVENT_CATALOG[["farm", "event_id", "event_label", "asset_id", "event_description"]])
VAR_TABLE = pd.DataFrame([{"symbol": k, "name": v["column"], "category": v["category"],
                           "description": v["description"], "unit": v["unit"]}
                          for k, v in symbol_map.items()])
display(VAR_TABLE)


## 2. Preprocessing

Following the base paper: (i) power-curve based cleaning of abnormal points, (ii) max-normalization, (iii) chronological 60/20/20 train/val/test split.


In [ ]:
# Preserve event order and use the dataset's train_test partition instead of an arbitrary global split.
data_df_clean = data_df.interpolate(limit_direction="both").ffill().bfill()
if data_df_clean.isna().any().any():
    raise ValueError("Selected CARE event still has missing sensor values after within-event imputation.")

train_test = sequence_meta.train_test.astype(str).str.lower().str.strip()
train_tokens = {"train", "training", "0"}
train_mask = train_test.isin(train_tokens)
if train_mask.sum() < 2 * CONFIG["WINDOW_SIZE"] or (~train_mask).sum() < CONFIG["WINDOW_SIZE"] + 1:
    raise ValueError("CARE train_test partition is too short for the configured window; choose another event or reduce WINDOW_SIZE.")

# Fit scaling exclusively on the provided training portion: no prediction-event leakage.
scale = data_df_clean.loc[train_mask].quantile(.99).abs().replace(0, 1.0)
data_norm = (data_df_clean / scale).clip(-5, 5).astype(np.float32)
train_full = data_norm.loc[train_mask].values
split = int(len(train_full) * CONFIG["TRAIN_RATIO"])
train_arr, val_arr = train_full[:split], train_full[split:]
test_arr = data_norm.loc[~train_mask].values
test_meta = sequence_meta.loc[~train_mask].reset_index(drop=True)
NUM_NODES, NODE_NAMES = data_norm.shape[1], list(data_norm.columns)
print("Dataset-provided partition -> train/val/test:", train_arr.shape, val_arr.shape, test_arr.shape)


### 2.1 Exploratory visualization (raw + cleaned signals, correlation structure, power curve)

In [ ]:

fig, axes = plt.subplots(int(np.ceil(NUM_NODES/4)), 4, figsize=(16, 3*int(np.ceil(NUM_NODES/4))))
axes = axes.flatten()
for i, col in enumerate(NODE_NAMES):
    axes[i].plot(data_norm[col].values[:2000], lw=0.6, color="tab:purple")
    axes[i].set_title(col, fontsize=8)
    axes[i].set_xticks([])
for j in range(i+1, len(axes)):
    axes[j].axis("off")
plt.suptitle("Normalized SCADA channels (first 2000 samples)")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 7))
corr_full = data_norm.corr()

sns.heatmap(
    corr_full,
    cmap="coolwarm",
    center=0,
    square=True,
    xticklabels=True,
    yticklabels=True,
    cbar_kws={"shrink": 0.7},
)
plt.title("Pearson correlation matrix of selected CARE SCADA channels")
plt.tight_layout()
plt.show()

# Optional real-data operating plot, only when metadata identifies both sensors.
wind_symbol = next(
    (key for key, value in symbol_map.items() if value["category"] == "wind"),
    None,
)
power_symbol = next(
    (key for key, value in symbol_map.items() if value["category"] == "power"),
    None,
)

if wind_symbol is not None and power_symbol is not None:
    plt.figure(figsize=(6, 5))
    plt.scatter(
        data_df_clean[wind_symbol],
        data_df_clean[power_symbol],
        s=3,
        alpha=0.2,
    )
    plt.xlabel(f"{wind_symbol} (wind-speed channel)")
    plt.ylabel(f"{power_symbol} (active-power channel)")
    plt.title("CARE operating relationship: wind speed vs. active power")
    plt.tight_layout()
    plt.show()
else:
    print("No metadata-confirmed wind-speed and active-power pair was selected; skipping operating plot.")

## 3. Physics-Guided Explicit Knowledge Graphs

We construct the same five explicit relationship types as the base paper (Section 4.2.1), generalized to work on the auto-detected node categories:

1. **Mechanical transmission** — energy-flow chain `wind -> rotor speed -> generator speed -> {active, reactive} power` (unidirectional).
2. **Temperature influence** — `ambient -> component temperatures` (unidirectional, ambient is an infinite heat source) and `nacelle <-> component` (bidirectional, nacelle mediates cross-component influence).
3. **Correlation** — Pearson |r| > threshold.
4. **Mutual information** — normalized MI > threshold.
5. **Causality** — sliding-window **linear Granger causality** (our declared, self-contained substitute for the paper's CORL algorithm; a drop-in replacement point for `gCastle`/`CORL` is documented below if you want to reproduce the paper's exact causal-discovery method).


In [ ]:

def empty_graph(n):
    return np.zeros((n, n), dtype=np.float32)


def build_mechanical_graph(symbol_map):
    n = len(symbol_map)
    idx = {k: i for i, k in enumerate(symbol_map)}
    A = empty_graph(n)
    wind_syms  = [k for k, v in symbol_map.items() if v["category"] == "wind"]
    speed_syms = [k for k, v in symbol_map.items() if v["category"] == "speed"]
    power_syms = [k for k, v in symbol_map.items() if v["category"] == "power"]
    # wind -> every speed node
    for w in wind_syms:
        for s in speed_syms:
            A[idx[w], idx[s]] = 1.0
    # speed nodes form a chain (rotor -> ... -> generator), approximated as fully forward-connected
    for i in range(len(speed_syms)):
        for j in range(i+1, len(speed_syms)):
            A[idx[speed_syms[i]], idx[speed_syms[j]]] = 1.0
    # last speed node(s) -> power
    for s in speed_syms:
        for p in power_syms:
            A[idx[s], idx[p]] = 1.0
    return A


def build_temperature_graph(symbol_map):
    n = len(symbol_map)
    idx = {key: i for i, key in enumerate(symbol_map)}
    A = empty_graph(n)

    power_syms = [
        key for key, value in symbol_map.items()
        if value["category"] == "power"
    ]
    temp_syms = [
        key for key, value in symbol_map.items()
        if value["category"] == "temperature"
    ]

    # Electrical/mechanical load causes component temperatures to rise.
    for power in power_syms:
        for temp in temp_syms:
            A[idx[power], idx[temp]] = 1.0

    # Connect temperatures only within documented physical subsystems.
    groups = {
        "gearbox": [],
        "generator": [],
        "converter": [],
    }

    for symbol in temp_syms:
        description = symbol_map[symbol]["description"].lower()

        if "gearbox" in description:
            groups["gearbox"].append(symbol)
        if "generator" in description:
            groups["generator"].append(symbol)
        if any(word in description for word in ["vcs", "igbt", "busbar", "choke", "vcp"]):
            groups["converter"].append(symbol)

    for members in groups.values():
        for source in members:
            for target in members:
                if source != target:
                    A[idx[source], idx[target]] = 1.0

    return A


def build_correlation_graph(data_arr, threshold):
    corr = np.corrcoef(data_arr.T)
    A = (np.abs(corr) >= threshold).astype(np.float32)
    np.fill_diagonal(A, 0.0)
    return A, corr


def build_mi_graph(data_arr, top_k=3, max_samples=4000):
    n = data_arr.shape[1]

    idx = np.random.choice(
        len(data_arr),
        size=min(max_samples, len(data_arr)),
        replace=False,
    )
    sub = data_arr[idx]

    mi_raw = np.zeros((n, n), dtype=np.float32)

    for target in range(n):
        scores = mutual_info_regression(
            sub,
            sub[:, target],
            random_state=SEED,
        )
        mi_raw[:, target] = scores

    mi_raw = (mi_raw + mi_raw.T) / 2.0
    np.fill_diagonal(mi_raw, 0.0)

    # Each sensor keeps its strongest information relationships.
    A = np.zeros((n, n), dtype=np.float32)

    for node in range(n):
        strongest = np.argsort(mi_raw[node])[::-1]

        kept = 0
        for neighbour in strongest:
            if neighbour == node or mi_raw[node, neighbour] <= 0:
                continue

            A[node, neighbour] = 1.0
            kept += 1

            if kept == top_k:
                break

    # MI is undirected: retain an edge chosen by either endpoint.
    A = np.maximum(A, A.T)
    np.fill_diagonal(A, 0.0)

    return A, mi_raw


In [ ]:

# ----------------------------------------------------------------------------
# 3.1  Causality graph: sliding-window linear Granger causality
#      (self-contained CORL substitute -- see markdown note above)
# ----------------------------------------------------------------------------
def granger_score(x_cause, y_effect, max_lag):
    '''F-test style score: variance reduction of predicting y_effect from
    its own lags + x_cause's lags, vs. y_effect's own lags alone.'''
    T = len(y_effect)
    if T <= 2*max_lag + 5:
        return 0.0
    Y = y_effect[max_lag:]
    X_restricted = np.column_stack([y_effect[max_lag-l:T-l] for l in range(1, max_lag+1)])
    X_full = np.column_stack([X_restricted] + [x_cause[max_lag-l:T-l] for l in range(1, max_lag+1)])

    def fit_resid(X, y):
        Xb = np.column_stack([np.ones(len(X)), X])
        beta, *_ = np.linalg.lstsq(Xb, y, rcond=None)
        resid = y - Xb @ beta
        return np.sum(resid**2)

    rss_r = fit_resid(X_restricted, Y)
    rss_f = fit_resid(X_full, Y)
    if rss_f <= 1e-10:
        return 1.0
    improvement = max(0.0, (rss_r - rss_f) / rss_r)
    return float(np.clip(improvement, 0.0, 1.0))


def build_causal_graph(data_arr, window, step, max_lag,
                       top_k=2, min_effect=0.0001):
    n = data_arr.shape[1]
    T = data_arr.shape[0]

    starts = list(range(0, max(1, T - window), step)) or [0]

    score_sum = np.zeros((n, n), dtype=np.float64)
    count = 0

    for start in starts:
        segment = data_arr[start:start + window]

        if len(segment) < 2 * max_lag + 10:
            continue

        for cause in range(n):
            for effect in range(n):
                if cause != effect:
                    score_sum[cause, effect] += granger_score(
                        segment[:, cause],
                        segment[:, effect],
                        max_lag,
                    )

        count += 1

    if count == 0:
        return empty_graph(n), empty_graph(n)

    causal_raw = (score_sum / count).astype(np.float32)
    A = np.zeros((n, n), dtype=np.float32)

    # For each effect sensor, retain only its strongest directed causes.
    for effect in range(n):
        ranked_causes = np.argsort(causal_raw[:, effect])[::-1]

        kept = 0
        for cause in ranked_causes:
            if cause == effect:
                continue
            if causal_raw[cause, effect] < min_effect:
                continue

            A[cause, effect] = 1.0
            kept += 1

            if kept == top_k:
                break

    np.fill_diagonal(A, 0.0)
    return A, causal_raw


In [ ]:

G_mech = build_mechanical_graph(symbol_map)
G_temp = build_temperature_graph(symbol_map)
G_corr, corr_raw = build_correlation_graph(train_arr, CONFIG["CORR_THRESH"])
G_mi, mi_raw = build_mi_graph(
    train_arr,
    top_k=CONFIG["MI_TOP_K"],
)
G_causal, causal_raw = build_causal_graph(
    train_arr,
    CONFIG["CAUSAL_WINDOW"],
    CONFIG["CAUSAL_STEP"],
    CONFIG["CAUSAL_MAXLAG"],
    top_k=CONFIG["CAUSAL_TOP_K"],
    min_effect=CONFIG["CAUSAL_MIN_EFFECT"],
)

EXPLICIT_GRAPHS = {
    "G1_mechanical":  G_mech,
    "G2_temperature": G_temp,
    "G3_correlation": G_corr,
    "G4_mutual_info": G_mi,
    "G5_causality":   G_causal,
}
for name, A in EXPLICIT_GRAPHS.items():
    print(f"{name:16s}: {int(A.sum())} directed edges out of {NUM_NODES*(NUM_NODES-1)} possible")


In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(22, 4.2))
for ax, (name, A) in zip(axes, EXPLICIT_GRAPHS.items()):
    sns.heatmap(A, ax=ax, cbar=False, cmap="Blues", xticklabels=False, yticklabels=False, square=True)
    ax.set_title(name)
plt.suptitle("Explicit knowledge graphs (adjacency matrices)")
plt.tight_layout()
plt.show()


## 4. Implicit Knowledge — Graph Structure Learning (GSL)

Following Yu et al. (RGSL) as cited by the base paper: learnable node embeddings, an inner-product kernel to produce edge-retention probabilities, and Gumbel-Softmax to sample a *discrete* implicit graph in a differentiable way (Eq. 6–7 of the base paper).


In [ ]:

class ImplicitGraphLearner(nn.Module):
    def __init__(self, num_nodes, emb_dim):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(num_nodes, emb_dim) * 0.1)

    def forward(self, tau=0.5, hard=False):
        theta = torch.sigmoid(self.embedding @ self.embedding.t())
        theta = theta.clamp(1e-4, 1 - 1e-4)
        g1 = -torch.log(-torch.log(torch.rand_like(theta) + 1e-10) + 1e-10)
        g2 = -torch.log(-torch.log(torch.rand_like(theta) + 1e-10) + 1e-10)
        logits = (torch.log(theta / (1 - theta)) + (g1 - g2)) / tau
        adj = torch.sigmoid(logits)
        adj = adj * (1 - torch.eye(adj.shape[0], device=adj.device))     # no self loops
        if hard:
            adj_hard = (adj > 0.5).float()
            adj = adj_hard + adj - adj.detach()                          # straight-through estimator
        return adj


## 5. Explicit–Implicit Knowledge Fusion (Quasi-Hard-Attention)

Reimplements Section 4.2.3: a Chebyshev-style first-order graph feature transform per graph, mean-pooled into an embedding, a scaled-dot-product attention over `{implicit, explicit_1..L}`, and a fusion rule that keeps the implicit graph as the *core* and adds explicit graphs weighted by their learned "**Implicit → Explicit_i**" attention (the quasi-hard part: `Im<->Im` fixed to 1, all other attention rows are ignored, matching Fig. 4 of the base paper).


In [ ]:

class GraphFeatureTransform(nn.Module):
    '''First-order Chebyshev-style graph propagation:  (I + D^-1/2 A D^-1/2) X W '''
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, X, A):
        deg = A.sum(-1) + 1e-6
        d_inv_sqrt = torch.diag(torch.pow(deg, -0.5))
        A_norm = d_inv_sqrt @ A @ d_inv_sqrt
        I = torch.eye(A.shape[0], device=A.device)
        return self.lin((I + A_norm) @ X)


class KnowledgeFusion(nn.Module):
    def __init__(self, num_nodes, in_dim, hidden_dim, num_explicit, emb_dim):
        super().__init__()
        self.implicit_learner = ImplicitGraphLearner(num_nodes, emb_dim)
        self.transform0 = GraphFeatureTransform(in_dim, hidden_dim)
        self.transforms = nn.ModuleList([GraphFeatureTransform(in_dim, hidden_dim)
                                          for _ in range(num_explicit)])
        self.Wq = nn.Linear(hidden_dim, hidden_dim)
        self.Wk = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, X_feat, explicit_graphs, tau=0.5, hard=False):
        A0 = self.implicit_learner(tau, hard)
        emb0 = self.transform0(X_feat, A0).mean(0, keepdim=True)

        embs, A_list = [emb0], []
        for i, Al in enumerate(explicit_graphs):
            embs.append(self.transforms[i](X_feat, Al).mean(0, keepdim=True))
            A_list.append(Al)
        M = torch.cat(embs, dim=0)                              # (1+L, hidden)

        Q, K = self.Wq(M), self.Wk(M)
        S = torch.softmax(Q @ K.t() / math.sqrt(M.shape[-1]), dim=-1)
        alpha = S[0, 1:]                                         # "Implicit -> Explicit_i" (quasi-hard)

        A_fused = A0.clone()
        for i, Al in enumerate(A_list):
            A_fused = A_fused + alpha[i] * Al
        A_fused = A_fused / (A_fused.max().clamp(min=1e-6))
        return A_fused, alpha, A0


## 6. Multivariate Time-Series Graph Neural Network (MTGNN)

Faithful reimplementation of Section 3.2 / Fig. 2: `1x1 conv -> [Dilated-Inception Temporal Conv -> Mix-Hop Graph Conv] x L -> skip connections -> output module`, with **true exponentially-growing dilation** (`d = 2^layer`) across the temporal-inception filters of size {2,3,6,7}.


In [ ]:

class DilatedInception(nn.Module):
    def __init__(self, cin, cout, dilation=1):
        super().__init__()
        self.kernel_set = [2, 3, 6, 7]
        cout_k = max(1, cout // len(self.kernel_set))
        self.convs = nn.ModuleList([
            nn.Conv2d(cin, cout_k, kernel_size=(1, k), dilation=(1, dilation)) for k in self.kernel_set
        ])

    def forward(self, x):
        outs = [c(x) for c in self.convs]
        min_len = min(o.shape[-1] for o in outs)
        outs = [o[..., -min_len:] for o in outs]
        return torch.cat(outs, dim=1)


class TemporalConvModule(nn.Module):
    def __init__(self, channels, dilation=1):
        super().__init__()
        self.filter_conv = DilatedInception(channels, channels, dilation)
        self.gate_conv = DilatedInception(channels, channels, dilation)

    def forward(self, x):
        f = torch.tanh(self.filter_conv(x))
        g = torch.sigmoid(self.gate_conv(x))
        return f * g


class GraphConvModule(nn.Module):
    '''Mix-hop propagation (Eq. 1-2 of base paper):
    H^(k) = beta*H_in + (1-beta) * A~ H^(k-1),   H_out = concat_k H^(k) W^(k)'''
    def __init__(self, channels, depth, beta):
        super().__init__()
        self.depth = depth
        self.beta = beta
        self.mlp = nn.Conv2d(channels * (depth + 1), channels, kernel_size=(1, 1))

    def forward(self, x, A):
        deg = A.sum(-1) + 1.0
        A_tilde = A / deg.unsqueeze(-1)
        H0 = x
        H = x
        outs = [H0]
        for _ in range(self.depth):
            H = self.beta * H0 + (1 - self.beta) * torch.einsum("nm,bcmt->bcnt", A_tilde, H)
            outs.append(H)
        return self.mlp(torch.cat(outs, dim=1))


class MTGNN(nn.Module):
    def __init__(self, num_nodes, seq_len, in_dim=1, channels=32, layers=2, gc_depth=2, beta=0.6, out_dim=1):
        super().__init__()
        self.start_conv = nn.Conv2d(in_dim, channels, kernel_size=(1, 1))
        self.tc_layers, self.gc_layers, self.skip_convs = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()

        cur_len = seq_len
        for l in range(layers):
            dilation = 2 ** l
            self.tc_layers.append(TemporalConvModule(channels, dilation))
            cur_len = cur_len - dilation * (7 - 1)
            assert cur_len > 0, "WINDOW_SIZE too small for chosen depth/dilation; increase CONFIG['WINDOW_SIZE']"
            self.gc_layers.append(GraphConvModule(channels, gc_depth, beta))
            self.skip_convs.append(nn.Conv2d(channels, channels, kernel_size=(1, cur_len)))

        self.end_conv1 = nn.Conv2d(channels, channels, kernel_size=(1, 1))
        self.end_conv2 = nn.Conv2d(channels, out_dim, kernel_size=(1, 1))
        self.act = nn.ReLU()

    def forward(self, x, A):
        x = self.start_conv(x)
        skip_sum = 0.0
        for tc, gc, sk in zip(self.tc_layers, self.gc_layers, self.skip_convs):
            residual = x
            x = tc(x)
            residual = residual[..., -x.shape[-1]:]
            x = gc(x, A) + residual
            skip_sum = skip_sum + sk(x)
        out = self.act(skip_sum)
        out = self.act(self.end_conv1(out))
        out = self.end_conv2(out)                 # (B, out_dim, N, 1)
        return out.squeeze(-1).squeeze(1)          # (B, N)


In [ ]:

# Sanity check: forward pass on random tensors before touching real data
_dummy_x = torch.randn(4, 1, NUM_NODES, CONFIG["WINDOW_SIZE"])
_dummy_A = torch.rand(NUM_NODES, NUM_NODES)
_test_model = MTGNN(num_nodes=NUM_NODES, seq_len=CONFIG["WINDOW_SIZE"],
                     channels=CONFIG["HIDDEN_DIM"], layers=CONFIG["NUM_TC_GC_LAYERS"],
                     gc_depth=CONFIG["GC_DEPTH"], beta=CONFIG["BETA"])
_out = _test_model(_dummy_x, _dummy_A)
print("MTGNN sanity check output shape:", _out.shape, "-> expected (4,", NUM_NODES, ")")
assert _out.shape == (4, NUM_NODES)
del _dummy_x, _dummy_A, _test_model, _out


## 7. Sequence Dataset & Full Digital-Twin Model (Fusion + MTGNN)

In [ ]:

class SCADASeqDataset(Dataset):
    def __init__(self, data_arr, window):
        self.data = data_arr.astype(np.float32)
        self.window = window

    def __len__(self):
        return max(0, len(self.data) - self.window)

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.window]                  # (window, N)
        y = self.data[idx + self.window]                       # (N,)
        x = x.T[np.newaxis, :, :]                               # (1, N, window)
        return torch.tensor(x), torch.tensor(y)


def make_loaders(window, batch_size):
    train_ds = SCADASeqDataset(train_arr, window)
    val_ds   = SCADASeqDataset(val_arr, window)
    test_ds  = SCADASeqDataset(test_arr, window)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True),
            DataLoader(val_ds, batch_size=batch_size, shuffle=False),
            DataLoader(test_ds, batch_size=batch_size, shuffle=False))


train_loader, val_loader, test_loader = make_loaders(CONFIG["WINDOW_SIZE"], CONFIG["BATCH_SIZE"])
print("Batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))


In [ ]:

class DigitalTwinModel(nn.Module):
    '''Full pipeline: GSL+fusion produces A_fused every forward pass,
    which conditions the MTGNN forecaster.'''
    def __init__(self, num_nodes, seq_len, explicit_graphs):
        super().__init__()
        self.fusion = KnowledgeFusion(num_nodes, in_dim=1, hidden_dim=CONFIG["HIDDEN_DIM"],
                                       num_explicit=len(explicit_graphs), emb_dim=CONFIG["EMB_DIM"])
        self.mtgnn = MTGNN(num_nodes=num_nodes, seq_len=seq_len, channels=CONFIG["HIDDEN_DIM"],
                            layers=CONFIG["NUM_TC_GC_LAYERS"], gc_depth=CONFIG["GC_DEPTH"], beta=CONFIG["BETA"])
        self.register_buffer("_dummy", torch.zeros(1))
        self.explicit_graphs = explicit_graphs

    def forward(self, x, tau=0.5, hard=False):
        X_feat = x.mean(dim=(0, 3)).transpose(0, 1)             # crude (N,1) node feature for fusion embedding
        A_fused, alpha, A0 = self.fusion(X_feat, self.explicit_graphs, tau, hard)
        pred = self.mtgnn(x, A_fused)
        return pred, A_fused, alpha


## 8. Generic Training / Evaluation Harness

In [ ]:

def rmse_r2(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return rmse, r2


def train_model(model, forward_fn, train_loader, val_loader, epochs, lr, patience, name="model"):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=CONFIG["WEIGHT_DECAY"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    loss_fn = nn.L1Loss()

    best_val, best_state, no_improve = float("inf"), None, 0
    history = {"train": [], "val": []}

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = forward_fn(model, xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = forward_fn(model, xb)
                val_losses.append(loss_fn(pred, yb).item())

        tr_loss, va_loss = np.mean(train_losses), np.mean(val_losses)
        history["train"].append(tr_loss); history["val"].append(va_loss)
        sched.step(va_loss)

        if va_loss < best_val - 1e-5:
            best_val, best_state, no_improve = va_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_improve += 1
        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"[{name}] epoch {epoch+1:3d}/{epochs} | train {tr_loss:.5f} | val {va_loss:.5f}")
        if no_improve >= patience:
            print(f"[{name}] early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


def evaluate_model(model, forward_fn, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = forward_fn(model, xb).cpu().numpy()
            preds.append(pred); trues.append(yb.numpy())
    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    rmse, r2 = rmse_r2(trues.flatten(), preds.flatten())
    return rmse, r2, preds, trues


## 9. Benchmark Models

Ten forecasting backbones are compared, mirroring Table 4 of the base paper:

- **No-graph:** LSTM, BiLSTM, CNN-LSTM, Transformer
- **Graph-augmented (unified under one `GraphTemporalForecaster` harness for a fair comparison protocol):** GCRN (GCN+GRU), GraphLSTM (GCN+LSTM), STGCN (GCN+CNN), GAT (learned attention over the prior-graph mask), NRI (learned relational inference gated by the prior-graph mask)
- **Proposed:** MTGNN with explicit-implicit fused graph


In [ ]:

# ---------------- No-graph baselines ----------------
class PlainLSTM(nn.Module):
    def __init__(self, num_nodes, hidden=64, bidirectional=False):
        super().__init__()
        self.rnn = nn.LSTM(input_size=num_nodes, hidden_size=hidden, batch_first=True, bidirectional=bidirectional)
        self.out = nn.Linear(hidden * (2 if bidirectional else 1), num_nodes)

    def forward(self, x, A=None):
        xin = x.squeeze(1).permute(0, 2, 1)               # (B,T,N)
        out, _ = self.rnn(xin)
        return self.out(out[:, -1])


class PlainCNNLSTM(nn.Module):
    def __init__(self, num_nodes, hidden=64):
        super().__init__()
        self.cnn = nn.Conv1d(num_nodes, hidden, kernel_size=3, padding=1)
        self.rnn = nn.LSTM(input_size=hidden, hidden_size=hidden, batch_first=True)
        self.out = nn.Linear(hidden, num_nodes)

    def forward(self, x, A=None):
        xin = x.squeeze(1)                                  # (B,N,T)
        h = F.relu(self.cnn(xin)).permute(0, 2, 1)
        out, _ = self.rnn(h)
        return self.out(out[:, -1])


class PlainTransformer(nn.Module):
    def __init__(self, num_nodes, hidden=64, seq_len=48):
        super().__init__()
        self.embed = nn.Linear(num_nodes, hidden)
        self.pos = nn.Parameter(torch.randn(1, seq_len, hidden) * 0.01)
        enc_layer = nn.TransformerEncoderLayer(d_model=hidden, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.out = nn.Linear(hidden, num_nodes)

    def forward(self, x, A=None):
        T = x.shape[-1]
        xin = x.squeeze(1).permute(0, 2, 1)
        h = self.embed(xin) + self.pos[:, :T]
        h = self.transformer(h)
        return self.out(h[:, -1])


In [ ]:

# ---------------- Graph-augmented unified harness ----------------
class SimpleGraphConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, x, A):                                # x: (B,T,N,F)
        deg = A.sum(-1) + 1.0
        A_norm = A / deg.unsqueeze(-1)
        x = torch.einsum("nm,btmf->btnf", A_norm, x)
        return self.lin(x)


class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a = nn.Linear(2 * out_dim, 1)

    def forward(self, x, A):                                # x: (B,N,F)
        Wh = self.W(x)
        B, N, O = Wh.shape
        Whi = Wh.unsqueeze(2).expand(B, N, N, O)
        Whj = Wh.unsqueeze(1).expand(B, N, N, O)
        e = F.leaky_relu(self.a(torch.cat([Whi, Whj], dim=-1)).squeeze(-1))
        mask_bias = torch.where(A > 0, torch.zeros_like(A), torch.full_like(A, -1e9))
        att = torch.softmax(e + mask_bias.unsqueeze(0), dim=-1)
        return torch.einsum("bnm,bmf->bnf", att, Wh)


class LearnedRelationGraph(nn.Module):
    '''NRI-style relation inference: an MLP over pairwise node embeddings
    predicts a soft adjacency, gated by the prior graph mask.'''
    def __init__(self, feat_dim, hidden=16):
        super().__init__()
        self.node_mlp = nn.Sequential(nn.Linear(feat_dim, hidden), nn.ReLU())
        self.edge_mlp = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x_last, prior_mask=None):
        h = self.node_mlp(x_last)
        B, N, H = h.shape
        hi = h.unsqueeze(2).expand(B, N, N, H)
        hj = h.unsqueeze(1).expand(B, N, N, H)
        e = self.edge_mlp(torch.cat([hi, hj], dim=-1)).squeeze(-1)
        A = torch.sigmoid(e)
        if prior_mask is not None:
            A = A * prior_mask.unsqueeze(0)
        return A


class GraphTemporalForecaster(nn.Module):
    def __init__(self, num_nodes, seq_len, backbone="lstm", mode="fixed_graph", hidden=32):
        super().__init__()
        self.mode, self.backbone_type = mode, backbone
        self.node_embed = nn.Linear(1, hidden)
        if mode == "fixed_graph":
            self.gconv = SimpleGraphConv(hidden, hidden)
        elif mode == "gat":
            self.gconv = GraphAttentionLayer(hidden, hidden)
        elif mode == "nri":
            self.relation_net = LearnedRelationGraph(hidden)
            self.gconv = SimpleGraphConv(hidden, hidden)

        if backbone in ["lstm", "gru"]:
            rnn_cls = nn.LSTM if backbone == "lstm" else nn.GRU
            self.rnn = rnn_cls(input_size=hidden, hidden_size=hidden, batch_first=True)
        elif backbone == "cnn":
            self.temporal_cnn = nn.Conv1d(hidden, hidden, kernel_size=3, padding=1)
        out_dim = hidden
        self.out = nn.Linear(out_dim, 1)

    def forward(self, x, A=None):
        B, C, N, T = x.shape
        xin = x.squeeze(1).permute(0, 2, 1).unsqueeze(-1)          # (B,T,N,1)
        h = self.node_embed(xin)                                    # (B,T,N,H)

        if self.mode == "fixed_graph":
            h = F.relu(self.gconv(h, A))
        elif self.mode == "gat":
            h_ = h.reshape(B * T, N, -1)
            h_ = F.relu(self.gconv(h_, A))
            h = h_.reshape(B, T, N, -1)
        elif self.mode == "nri":
            A_learned = self.relation_net(h[:, -1], prior_mask=A)
            h_ = h.reshape(B * T, N, -1)
            A_rep = A_learned.unsqueeze(1).expand(B, T, N, N).reshape(B * T, N, N)
            h_ = torch.einsum("bnm,bmf->bnf", A_rep, h_)
            h = F.relu(h_).reshape(B, T, N, -1)

        h = h.permute(0, 2, 1, 3).reshape(B * N, T, -1)              # treat each node as a sequence
        if self.backbone_type in ["lstm", "gru"]:
            out, _ = self.rnn(h)
            out = out[:, -1]
        elif self.backbone_type == "cnn":
            out = self.temporal_cnn(h.permute(0, 2, 1)).permute(0, 2, 1)[:, -1]
        pred = self.out(out).reshape(B, N)
        return pred


## 10. Full Experiment Grid — Modeling Accuracy Ablation (replicates Table 4)

We now train **every model x every graph condition** combination that applies, exactly mirroring the paper's ablation:
`No Graph, G1 (mechanical), G2 (temperature), G3 (correlation), G4 (mutual information), G5 (causality), G_fused`.


In [ ]:

def to_tensor_graph(A_np):
    return torch.tensor(A_np, dtype=torch.float32, device=device)

EXPLICIT_TENSORS = {k: to_tensor_graph(v) for k, v in EXPLICIT_GRAPHS.items()}

def build_baseline(name, graph_key):
    seq_len = CONFIG["WINDOW_SIZE"]
    if name == "LSTM":         return PlainLSTM(NUM_NODES)
    if name == "BiLSTM":       return PlainLSTM(NUM_NODES, bidirectional=True)
    if name == "CNN-LSTM":     return PlainCNNLSTM(NUM_NODES)
    if name == "Transformer":  return PlainTransformer(NUM_NODES, seq_len=seq_len)
    if name == "GraphLSTM":    return GraphTemporalForecaster(NUM_NODES, seq_len, backbone="lstm", mode="fixed_graph")
    if name == "GCRN":         return GraphTemporalForecaster(NUM_NODES, seq_len, backbone="gru",  mode="fixed_graph")
    if name == "STGCN":        return GraphTemporalForecaster(NUM_NODES, seq_len, backbone="cnn",  mode="fixed_graph")
    if name == "GAT":          return GraphTemporalForecaster(NUM_NODES, seq_len, backbone="lstm", mode="gat")
    if name == "NRI":          return GraphTemporalForecaster(NUM_NODES, seq_len, backbone="lstm", mode="nri")
    raise ValueError(name)

NO_GRAPH_MODELS   = ["LSTM", "BiLSTM", "CNN-LSTM", "Transformer"]
GRAPH_BASED_MODELS = ["GraphLSTM", "GAT", "NRI", "GCRN", "STGCN"]
GRAPH_CONDITIONS  = ["No Graph"] + list(EXPLICIT_GRAPHS.keys()) + ["G_fused"]

FAST_EPOCHS = min(CONFIG["EPOCHS"], 18)     # keep the full ablation grid tractable

results = []
RUN_BASELINES = os.getenv("RUN_BASELINES", "0").strip() == "1"
if RUN_BASELINES:
    for name in NO_GRAPH_MODELS:
        model = build_baseline(name, None)
        fwd = lambda m, xb: m(xb)
        model, _ = train_model(model, fwd, train_loader, val_loader, FAST_EPOCHS, CONFIG["LR"],
                                patience=5, name=f"{name}/NoGraph")
        rmse, r2, _, _ = evaluate_model(model, fwd, test_loader)
        results.append({"model": name, "graph": "No Graph", "rmse": rmse, "r2": r2})
        print(f"{name:12s} | No Graph  | RMSE={rmse:.4f} R2={r2:.4f}")


    for name in GRAPH_BASED_MODELS:
        for gkey, A_t in EXPLICIT_TENSORS.items():
            model = build_baseline(name, gkey)
            fwd = lambda m, xb, A=A_t: m(xb, A)
            model, _ = train_model(model, fwd, train_loader, val_loader, FAST_EPOCHS, CONFIG["LR"],
                                    patience=5, name=f"{name}/{gkey}")
            rmse, r2, _, _ = evaluate_model(model, fwd, test_loader)
            results.append({"model": name, "graph": gkey, "rmse": rmse, "r2": r2})
            print(f"{name:12s} | {gkey:16s} | RMSE={rmse:.4f} R2={r2:.4f}")


    # MTGNN with each single explicit graph (graph fixed, no fusion) + no-graph identity variant
    identity_A = to_tensor_graph(np.ones((NUM_NODES, NUM_NODES), dtype=np.float32) - np.eye(NUM_NODES, dtype=np.float32))

    for gkey, A_t in list(EXPLICIT_TENSORS.items()) + [("No Graph", identity_A)]:
        mtgnn = MTGNN(num_nodes=NUM_NODES, seq_len=CONFIG["WINDOW_SIZE"], channels=CONFIG["HIDDEN_DIM"],
                      layers=CONFIG["NUM_TC_GC_LAYERS"], gc_depth=CONFIG["GC_DEPTH"], beta=CONFIG["BETA"])
        fwd = lambda m, xb, A=A_t: m(xb, A)
        mtgnn, _ = train_model(mtgnn, fwd, train_loader, val_loader, FAST_EPOCHS, CONFIG["LR"],
                                patience=5, name=f"MTGNN/{gkey}")
        rmse, r2, _, _ = evaluate_model(mtgnn, fwd, test_loader)
        results.append({"model": "MTGNN", "graph": gkey, "rmse": rmse, "r2": r2})
        print(f"MTGNN        | {gkey:16s} | RMSE={rmse:.4f} R2={r2:.4f}")

else:
    print("Skipping expensive baseline grid for first run. Set RUN_BASELINES=1 to run the full ablation.")


In [ ]:
# Baseline loops are guarded by RUN_BASELINES in cell 36 for a fast first run.


In [ ]:
# MTGNN graph ablation loop is guarded by RUN_BASELINES in cell 36.


### 10.1 Train the full **Digital-Twin model** (learned fusion + MTGNN) — this is the proposed pipeline

In [ ]:

digital_twin = DigitalTwinModel(NUM_NODES, CONFIG["WINDOW_SIZE"], list(EXPLICIT_TENSORS.values()))
fwd_twin = lambda m, xb: m(xb, tau=CONFIG["GUMBEL_TAU"], hard=False)[0]

digital_twin, dt_history = train_model(digital_twin, fwd_twin, train_loader, val_loader,
                                        CONFIG["EPOCHS"], CONFIG["LR"], CONFIG["PATIENCE"],
                                        name="DigitalTwin(Fusion+MTGNN)")
rmse_fused, r2_fused, dt_preds, dt_trues = evaluate_model(digital_twin, fwd_twin, test_loader)
results.append({"model": "MTGNN", "graph": "G_fused (learned)", "rmse": rmse_fused, "r2": r2_fused})
print(f"\\nMTGNN + learned fusion | RMSE={rmse_fused:.4f} | R2={r2_fused:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(dt_history["train"], label="train")
plt.plot(dt_history["val"], label="val")
plt.xlabel("epoch"); plt.ylabel("L1 loss"); plt.legend()
plt.title("Digital-Twin model training curves")
plt.tight_layout(); plt.show()


In [ ]:

results_df = pd.DataFrame(results)
pivot_rmse = results_df.pivot_table(index="model", columns="graph", values="rmse")
pivot_r2   = results_df.pivot_table(index="model", columns="graph", values="r2")

order_cols = ["No Graph"] + list(EXPLICIT_GRAPHS.keys()) + ["G_fused", "G_fused (learned)"]
order_cols = [c for c in order_cols if c in pivot_rmse.columns]
pivot_rmse = pivot_rmse[order_cols]
pivot_r2 = pivot_r2[order_cols]

print("=== RMSE (lower is better) ===")
display(pivot_rmse.round(4))
print("\\n=== R^2 (higher is better) ===")
display(pivot_r2.round(4))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(pivot_rmse, annot=True, fmt=".3f", cmap="rocket_r", ax=axes[0])
axes[0].set_title("RMSE across model x graph-condition")
sns.heatmap(pivot_r2, annot=True, fmt=".3f", cmap="viridis", ax=axes[1])
axes[1].set_title("R$^2$ across model x graph-condition")
plt.tight_layout(); plt.show()


### Note: duplicate training removed for faster local execution

The proposed Digital-Twin model was trained and evaluated in Section 10.1 above. This duplicate cell was intentionally removed to avoid spending time training the same model twice.


## 11. Anomaly Degree (Section 4.3 of base paper)

For each variable, the residual distribution of the trained digital-twin model on **normal-condition validation data** is fitted with a Gaussian `N(mu, sigma^2)`. The anomaly degree of an observed residual is then

`degree(z) = 1 - exp(-0.5 * z^2)`,  where `z = (residual - mu) / sigma`

so `degree=0` at the distribution's center and `degree -> 1` as the residual becomes an extreme outlier — reproducing the bell-shaped mapping shown in Fig. 6 of the base paper, inverted to express *anomalousness* rather than *typicality*.

We additionally attach **Monte-Carlo Dropout epistemic uncertainty bands** around each anomaly-degree estimate (a capability the base paper does not include) so that root-cause claims can be reported with a confidence level.


In [ ]:

def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

def compute_residual_stats(model, fwd_fn, loader):
    model.eval()
    residuals = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = fwd_fn(model, xb).cpu().numpy()
            residuals.append(pred - yb.numpy())
    residuals = np.concatenate(residuals, axis=0)
    mu = residuals.mean(0)
    sigma = residuals.std(0) + 1e-6
    return mu, sigma, residuals

mu_res, sigma_res, val_residuals = compute_residual_stats(digital_twin, fwd_twin, val_loader)

def anomaly_degree(residual, mu, sigma):
    z = (residual - mu) / sigma
    return np.clip(1 - np.exp(-0.5 * z ** 2), 0, 1)

plt.figure(figsize=(7, 4))
for i in range(min(4, NUM_NODES)):
    sns.kdeplot(val_residuals[:, i], label=NODE_NAMES[i])
plt.title("Validation residual distributions (normal condition)")
plt.xlabel("residual"); plt.legend(fontsize=7)
plt.tight_layout(); plt.show()


In [ ]:

def mc_dropout_anomaly_degree(model, fwd_fn, x_window, y_true, mu, sigma, passes=20):
    '''Runs several stochastic forward passes (dropout active) to obtain an
    epistemic-uncertainty band around the anomaly degree for one timestep.'''
    was_training = model.training
    model.eval()
    enable_mc_dropout(model)
    preds = []
    with torch.no_grad():
        for _ in range(passes):
            pred = fwd_fn(model, x_window.to(device)).cpu().numpy()
            preds.append(pred)
    preds = np.stack(preds, axis=0)             # (passes, B, N)
    residual_samples = preds - y_true.numpy()[None, ...]
    degree_samples = anomaly_degree(residual_samples, mu, sigma)
    model.train(was_training)
    return degree_samples.mean(0), degree_samples.std(0)             # (B,N) each


## 12. Fault Propagation Analysis & Root Cause Localization (Section 4.4)

For a fault window we compute the per-node anomaly-degree trajectory, visualize it as a sequence of colored graph snapshots (as in Fig. 11–13 / 17–18 of the base paper), then localize the root cause with **three complementary, cross-validated methods**:

1. **Earliest-onset backtracking** (the paper's method) — the node whose anomaly degree first crosses a threshold.
2. **Propagation-delay graph + centrality ranking** *(novel addition)* — build a directed graph where an edge `i -> j` exists if node `i`'s anomaly degree series leads node `j`'s (via cross-correlation lag), then rank nodes by out-degree / eigenvector centrality as "sources" of the propagating anomaly.
3. **Confidence-scored aggregation** *(novel addition)* — combine (1) and (2), weighted by the Monte-Carlo-Dropout uncertainty of each node's anomaly-degree estimate, into a single Root-Cause Confidence Score.


In [ ]:
def get_fault_window(window_before=120, window_len=250):
    """Locate the real event using event_info IDs first, timestamps second."""
    start_id = str(active_event.event_start_id)
    match = np.flatnonzero(test_meta.id.astype(str).eq(start_id).to_numpy()) if "id" in test_meta else np.array([])
    if len(match) == 0:
        event_time = pd.to_datetime(active_event.event_start, errors="coerce")
        match = np.flatnonzero(test_meta.time_stamp.ge(event_time).to_numpy())
    start_at = int(match[0]) if len(match) else max(0, len(test_arr) - window_len)
    return max(CONFIG["WINDOW_SIZE"], start_at - window_before), min(len(test_arr), start_at + window_len), None

f_start, f_end, TRUE_ROOT = get_fault_window()
print(f"Real event analysis window: test indices [{f_start}, {f_end})")
print("Logged root-cause description:", active_event.event_description)


In [ ]:

def compute_anomaly_trajectory(model, fwd_fn, data_arr, start, end, window, mu, sigma):
    '''Slides the trained digital twin across [start,end) and returns the
    (T, N) anomaly-degree trajectory plus MC-dropout uncertainty.'''
    degrees, uncertainties = [], []
    model.eval()
    idxs = range(max(window, start), end)
    for t in idxs:
        x = data_arr[t - window: t]
        y = data_arr[t]
        xb = torch.tensor(x.T[np.newaxis, np.newaxis, :, :], dtype=torch.float32)
        yb = torch.tensor(y[np.newaxis, :], dtype=torch.float32)
        deg_mean, deg_std = mc_dropout_anomaly_degree(model, fwd_fn, xb, yb, mu, sigma,
                                                       passes=CONFIG["MC_DROPOUT_PASSES"])
        degrees.append(deg_mean[0]); uncertainties.append(deg_std[0])
    return np.array(degrees), np.array(uncertainties), list(idxs)


anomaly_traj, anomaly_unc, traj_idxs = compute_anomaly_trajectory(
    digital_twin, fwd_twin, test_arr, f_start, f_end, CONFIG["WINDOW_SIZE"], mu_res, sigma_res)
print("Anomaly trajectory shape (T, N):", anomaly_traj.shape)


In [ ]:

plt.figure(figsize=(11, 5))
for i in range(NUM_NODES):
    plt.plot(anomaly_traj[:, i], label=NODE_NAMES[i], alpha=0.8)
plt.axhline(CONFIG["ANOMALY_THRESH"], color="k", ls="--", lw=1, label="anomaly threshold")
plt.xlabel("timestep (relative to fault window)"); plt.ylabel("anomaly degree")
plt.title("Fault propagation: anomaly degree of every node over time")
plt.legend(fontsize=6, ncol=3)
plt.tight_layout(); plt.show()


In [ ]:

def plot_propagation_snapshots(A_graph_np, anomaly_traj, node_names, n_snapshots=4, seed=SEED):
    G = nx.from_numpy_array(A_graph_np, create_using=nx.DiGraph)
    mapping = {i: node_names[i] for i in range(len(node_names))}
    G = nx.relabel_nodes(G, mapping)
    pos = nx.spring_layout(G, seed=seed)

    T = anomaly_traj.shape[0]
    snap_idxs = np.linspace(0, T - 1, n_snapshots).astype(int)
    fig, axes = plt.subplots(1, n_snapshots, figsize=(5 * n_snapshots, 5))
    for ax, t in zip(axes, snap_idxs):
        degrees = anomaly_traj[t]
        colors = [degrees[node_names.index(n)] for n in G.nodes()]
        nodes = nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, cmap="coolwarm",
                                        vmin=0, vmax=1, node_size=500)
        nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.15, arrows=True, arrowsize=6)
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=7)
        ax.set_title(f"t = {t}")
        ax.axis("off")
    cbar = fig.colorbar(nodes, ax=axes, shrink=0.7)
    cbar.set_label("anomaly degree")
    plt.suptitle("Fault propagation snapshots on the fused graph")
    plt.show()

A_fused_np = digital_twin.fusion.forward(
    torch.zeros(NUM_NODES, 1, device=device),   # dummy features: we only need A_fused topology here
    list(EXPLICIT_TENSORS.values()), tau=CONFIG["GUMBEL_TAU"], hard=True
)[0].detach().cpu().numpy()

plot_propagation_snapshots(A_fused_np, anomaly_traj, NODE_NAMES)


### 12.1 Root-cause localization method 1 — Earliest-onset backtracking (paper's method)

In [ ]:

def earliest_onset_root_cause(anomaly_traj, node_names, threshold):
    onset_times = []
    for i in range(anomaly_traj.shape[1]):
        crossing = np.where(anomaly_traj[:, i] >= threshold)[0]
        onset_times.append(crossing[0] if len(crossing) else np.inf)
    onset_times = np.array(onset_times)
    order = np.argsort(onset_times)
    ranking = [(node_names[i], int(onset_times[i]) if np.isfinite(onset_times[i]) else None)
               for i in order if np.isfinite(onset_times[i])]
    return ranking

onset_ranking = earliest_onset_root_cause(anomaly_traj, NODE_NAMES, CONFIG["ANOMALY_THRESH"])
print("Earliest-onset ranking (candidate root cause first):")
for rank, (name, t) in enumerate(onset_ranking[:8], 1):
    print(f"  {rank}. {name:6s} first crosses threshold at t={t}")


### 12.2 Root-cause localization method 2 — Propagation-delay graph + centrality (novel)

In [ ]:

def cross_corr_lag(x, y, max_lag=40):
    '''Returns (best_lag, best_corr). Positive lag means x leads y.'''
    best_lag, best_corr = 0, -np.inf
    for lag in range(0, max_lag + 1):
        if lag == 0:
            xs, ys = x, y
        else:
            xs, ys = x[:-lag], y[lag:]
        if len(xs) < 10 or np.std(xs) < 1e-8 or np.std(ys) < 1e-8:
            continue
        c = np.corrcoef(xs, ys)[0, 1]
        if c > best_corr:
            best_corr, best_lag = c, lag
    return best_lag, best_corr


def build_propagation_graph(anomaly_traj, node_names, max_lag=40, corr_thresh=0.5):
    n = anomaly_traj.shape[1]
    P = nx.DiGraph()
    P.add_nodes_from(node_names)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            lag, corr = cross_corr_lag(anomaly_traj[:, i], anomaly_traj[:, j], max_lag)
            if lag > 0 and corr >= corr_thresh:
                P.add_edge(node_names[i], node_names[j], weight=corr, lag=lag)
    return P

prop_graph = build_propagation_graph(anomaly_traj, NODE_NAMES)
centrality = nx.out_degree_centrality(prop_graph)
try:
    eig_centrality = nx.eigenvector_centrality(prop_graph, max_iter=1000)
except Exception:
    eig_centrality = {k: 0.0 for k in prop_graph.nodes()}

centrality_ranking = sorted(centrality.items(), key=lambda kv: -(kv[1] + eig_centrality.get(kv[0], 0)))
print("Propagation-delay centrality ranking (candidate root cause first):")
for rank, (name, score) in enumerate(centrality_ranking[:8], 1):
    print(f"  {rank}. {name:6s} out-degree centrality={score:.3f}  eigenvector={eig_centrality.get(name,0):.3f}")

plt.figure(figsize=(6, 6))
pos = nx.spring_layout(prop_graph, seed=SEED)
node_sizes = [3000 * centrality.get(n, 0) + 300 for n in prop_graph.nodes()]
nx.draw(prop_graph, pos, with_labels=True, node_color="lightsteelblue",
        node_size=node_sizes, arrowsize=12, font_size=8)
edge_labels = {(u, v): f"lag={d['lag']}" for u, v, d in prop_graph.edges(data=True)}
nx.draw_networkx_edge_labels(prop_graph, pos, edge_labels=edge_labels, font_size=6)
plt.title("Inferred fault-propagation-delay graph (edge = leads-by lag)")
plt.tight_layout(); plt.show()


### 12.3 Root-cause localization method 3 — Confidence-scored aggregation (novel)

In [ ]:

def aggregate_root_cause_score(onset_ranking, centrality_ranking, anomaly_unc, node_names):
    n = len(node_names)
    onset_rank = {name: r for r, (name, _) in enumerate(onset_ranking)}
    centrality_rank = {name: r for r, (name, _) in enumerate(centrality_ranking)}
    mean_uncertainty = anomaly_unc.mean(axis=0)                       # (N,)
    scores = []
    for i, name in enumerate(node_names):
        r1 = onset_rank.get(name, n)
        r2 = centrality_rank.get(name, n)
        combined_rank = 0.5 * r1 + 0.5 * r2
        confidence = 1.0 / (1.0 + mean_uncertainty[i])                # lower epistemic uncertainty -> higher confidence
        score = confidence / (1.0 + combined_rank)
        scores.append((name, score, confidence, combined_rank))
    scores.sort(key=lambda t: -t[1])
    return scores

final_scores = aggregate_root_cause_score(onset_ranking, centrality_ranking, anomaly_unc, NODE_NAMES)
score_df = pd.DataFrame(final_scores, columns=["node", "root_cause_score", "confidence", "combined_rank"])
print("=== Final Root-Cause Confidence Ranking ===")
display(score_df.head(8).round(4))

predicted_root = score_df.iloc[0]["node"]
print(f"\\nPredicted root cause: {predicted_root}")
if TRUE_ROOT is not None:
    hit = predicted_root == TRUE_ROOT
    top3_hit = TRUE_ROOT in score_df["node"].values[:3]
    print(f"Sensor-level ground truth is unavailable for this CARE event: {TRUE_ROOT}")
    print(f"Exact match: {hit} | Top-3 match: {top3_hit}")


In [ ]:

plt.figure(figsize=(7, 4))
sns.barplot(data=score_df.head(10), x="root_cause_score", y="node", palette="rocket")
plt.title("Root-cause confidence score (higher = stronger root-cause candidate)")
plt.tight_layout(); plt.show()


## 13. Real-event root-cause report

CARE provides a component-level `event_description` for some anomaly events, but it does not provide a sensor-level ground-truth root node. Therefore this notebook reports the model's ranked sensor candidates alongside the logged description; it does **not** claim synthetic Top-1/Top-3 accuracy.


In [ ]:
ranked_root_causes = score_df.rename(columns={"node": "sensor"}).copy()
ranked_root_causes["sensor_name"] = ranked_root_causes.sensor.map(lambda s: symbol_map[s]["column"])
ranked_root_causes["description"] = ranked_root_causes.sensor.map(lambda s: symbol_map[s]["description"])
ranked_root_causes["unit"] = ranked_root_causes.sensor.map(lambda s: symbol_map[s]["unit"])
display(ranked_root_causes.head(10))
print("CARE logged event description:", active_event.event_description)
print("Interpretation: ranked sensors are model hypotheses; the event description is the available component-level reference.")


## 14. Summary

This run is tied to the actual CARE/Kaggle schema: it discovers farms through `event_info.csv`, maps each event explicitly to its CSV, retains the complete event catalog, uses `feature_description.csv` to identify average sensor channels, honors `train_test`, and anchors fault-window localization in the event registry. It contains no synthetic simulator, filename-based fault detection, or arbitrary 12-file/single-turbine reduction.


In [ ]:

with torch.no_grad():
    X_feat_dummy = torch.zeros(NUM_NODES, 1, device=device)
    _, alpha_weights, _ = digital_twin.fusion(X_feat_dummy, list(EXPLICIT_TENSORS.values()),
                                               tau=CONFIG["GUMBEL_TAU"], hard=False)
alpha_weights = alpha_weights.detach().cpu().numpy()

plt.figure(figsize=(6, 4))
sns.barplot(x=list(EXPLICIT_GRAPHS.keys()), y=alpha_weights, palette="mako")
plt.ylabel("Implicit -> Explicit_i attention weight")
plt.title("Learned reliance on each explicit knowledge graph")
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()


In [ ]:

def gradient_saliency(model, fwd_fn, x_sample):
    model.eval()
    x_sample = x_sample.clone().to(device).requires_grad_(True)
    pred = fwd_fn(model, x_sample)
    target = pred.sum()
    target.backward()
    saliency = x_sample.grad.detach().cpu().numpy()[0, 0]     # (N, T)
    return np.abs(saliency)

sample_x, _ = next(iter(test_loader))
sal = gradient_saliency(digital_twin, fwd_twin, sample_x[:1])

plt.figure(figsize=(9, 5))
sns.heatmap(sal, cmap="magma", yticklabels=NODE_NAMES, xticklabels=False)
plt.xlabel("time steps in input window (most recent -> right)")
plt.title("Gradient saliency: which (node, timestep) most influence the next-step forecast")
plt.tight_layout(); plt.show()


In [ ]:

def run_seeded_experiment(seed, best_baseline_name="GAT", best_baseline_graph="G3_correlation"):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    dt = DigitalTwinModel(NUM_NODES, CONFIG["WINDOW_SIZE"], list(EXPLICIT_TENSORS.values()))
    fwd_dt = lambda m, xb: m(xb, tau=CONFIG["GUMBEL_TAU"], hard=False)[0]
    dt, _ = train_model(dt, fwd_dt, train_loader, val_loader, epochs=12, lr=CONFIG["LR"],
                         patience=4, name=f"seed{seed}/DigitalTwin")
    rmse_dt, _, _, _ = evaluate_model(dt, fwd_dt, test_loader)

    base = build_baseline(best_baseline_name, best_baseline_graph)
    A_t = EXPLICIT_TENSORS[best_baseline_graph]
    fwd_base = lambda m, xb, A=A_t: m(xb, A)
    base, _ = train_model(base, fwd_base, train_loader, val_loader, epochs=12, lr=CONFIG["LR"],
                           patience=4, name=f"seed{seed}/{best_baseline_name}")
    rmse_base, _, _, _ = evaluate_model(base, fwd_base, test_loader)
    return rmse_dt, rmse_base

SEEDS = [1, 2, 3]
paired_results = [run_seeded_experiment(s) for s in SEEDS]
dt_rmses = [r[0] for r in paired_results]
base_rmses = [r[1] for r in paired_results]

t_stat, p_value = stats.ttest_rel(base_rmses, dt_rmses)
print("Digital-Twin (fused MTGNN) RMSE across seeds:", np.round(dt_rmses, 4))
print("Best-baseline RMSE across seeds:            ", np.round(base_rmses, 4))
print(f"Paired t-test: t={t_stat:.3f}, p={p_value:.4f} "
      f"({'significant' if p_value < 0.05 else 'not significant'} improvement at alpha=0.05)")


In [ ]:

from pathlib import Path
import torch
import json

# Find the project root (the notebook is inside "notebooks")
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save every trained PyTorch model exposed as a top-level variable
saved_models = []

for name, obj in list(globals().items()):
    if isinstance(obj, torch.nn.Module):
        filename = "".join(
            c if c.isalnum() or c in "-_" else "_"
            for c in name
        )
        path = MODEL_DIR / f"{filename}.pth"
        torch.save(obj.state_dict(), path)
        saved_models.append({
            "variable": name,
            "file": path.name,
            "class": type(obj).__name__
        })
        print(f"Saved: {path}")

# Record an inventory of saved models
with open(MODEL_DIR / "model_inventory.json", "w", encoding="utf-8") as f:
    json.dump(saved_models, f, indent=2)

print(f"\nModels saved: {len(saved_models)}")
print(f"Save location: {MODEL_DIR.resolve()}")

if not saved_models:
    print("No top-level PyTorch models found. We need to identify the notebook's model variables.")


## 15. Extension points

For a different real case study, set `TARGET_FARM` and `TARGET_EVENT_ID` to values shown in the Section 1 catalog. Keep the event-level separation and use the registry timestamps/IDs when evaluating detection lead time.


## 16. Export results for the VS Code dashboard
Run this cell after all analysis cells finish successfully. The dashboard uses these real outputs.


In [ ]:
# Add this as a NEW code cell at the END of the existing notebook, after cells
# that define anomaly_traj, anomaly_unc, traj_idxs, score_df, G_causal,
# NODE_NAMES, symbol_map, active_event, CONFIG, test_arr, test_meta.
# It exports results from the actual trained notebook; it does not create fake data.

from pathlib import Path
import json
import numpy as np
import pandas as pd
# Save results in a separate folder for the current event
EVENT_ID = int(active_event["event_id"])

OUT = Path("artifacts") / f"event_{EVENT_ID}"
OUT.mkdir(parents=True, exist_ok=True)

# Event metadata
def scalar(v):
    try:
        if pd.isna(v): return None
    except Exception:
        pass
    if hasattr(v, "item"):
        try: return v.item()
        except Exception: pass
    return str(v)

meta = {
    "farm": scalar(active_event.get("farm", None)),
    "event_id": scalar(active_event.get("event_id", None)),
    "event_label": scalar(active_event.get("event_label", None)),
    "asset_id": scalar(active_event.get("asset_id", None)),
    "event_description": scalar(active_event.get("event_description", None)),
    "anomaly_threshold": float(CONFIG["ANOMALY_THRESH"]),
    "window_size": int(CONFIG["WINDOW_SIZE"]),
    "note": "Scores and rankings are model evidence, not calibrated probabilities."
}
(OUT/"metadata.json").write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")

sensors = pd.DataFrame([{
    "sensor": str(k),
    "name": str(v.get("column", k)),
    "category": str(v.get("category", "unknown")),
    "description": str(v.get("description", "")),
    "unit": str(v.get("unit", ""))
} for k,v in symbol_map.items()])
sensors.to_csv(OUT/"sensors.csv", index=False)

# Align the trajectory with test indices returned by compute_anomaly_trajectory.
# anomaly_traj rows correspond to traj_idxs; preserve the notebook's actual data.
T = len(traj_idxs)
ts = pd.DataFrame({"time_index": np.asarray(traj_idxs, dtype=int)})
for i, sensor in enumerate(NODE_NAMES):
    ts[f"anomaly__{sensor}"] = anomaly_traj[:T, i]
    ts[f"uncertainty__{sensor}"] = anomaly_unc[:T, i]

# Export actual values. Forecasts are added only if the notebook has a compatible
# per-timestep prediction array; do not silently invent forecasts.
for i, sensor in enumerate(NODE_NAMES):
    vals = []
    for ti in traj_idxs:
        if 0 <= int(ti) < len(test_arr):
            vals.append(float(test_arr[int(ti), i]))
        else:
            vals.append(np.nan)
    ts[f"actual__{sensor}"] = vals

# If dt_preds/dt_trues are batch outputs, their alignment is not assumed here.
# Add predictions only after validating exact index alignment in your experiment.
ts.to_csv(OUT/"timeseries.csv", index=False)

roots = score_df.rename(columns={"node":"sensor"}).copy()
roots.to_csv(OUT/"root_causes.csv", index=False)

# Export explicit graph edges and the learned fused graph when available.
edge_rows = []
def add_edges(graph_name, matrix, node_names):
    arr = np.asarray(matrix)
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            w = float(arr[i,j])
            if i != j and np.isfinite(w) and w > 0:
                edge_rows.append({"graph_type": graph_name, "source": node_names[i],
                                  "target": node_names[j], "weight": w})
for graph_name, matrix in EXPLICIT_GRAPHS.items():
    add_edges(graph_name, matrix, NODE_NAMES)
try:
    add_edges("Propagation", nx.to_numpy_array(P_graph, nodelist=NODE_NAMES, weight="weight"), NODE_NAMES)
except Exception:
    try:
        add_edges("Propagation", nx.to_numpy_array(propagation_graph, nodelist=NODE_NAMES, weight="weight"), NODE_NAMES)
    except Exception:
        pass
pd.DataFrame(edge_rows, columns=["graph_type","source","target","weight"]).to_csv(OUT/"graph_edges.csv", index=False)

# Existing gradient saliency (not SHAP) is exported when `sal` exists and matches
# the expected (sensor, time-step) shape.
try:
    sal_arr = np.asarray(sal)
    if sal_arr.ndim == 2 and sal_arr.shape[0] == len(NODE_NAMES):
        sal_df = pd.DataFrame([
            {"sensor": NODE_NAMES[i], "time_step": j, "importance": float(sal_arr[i,j])}
            for i in range(sal_arr.shape[0]) for j in range(sal_arr.shape[1])
        ])
        sal_df.to_csv(OUT/"saliency.csv", index=False)
except Exception as exc:
    print("Saliency export skipped:", exc)

print(f"Dashboard artifacts exported to: {OUT.resolve()}")
print("Files:", sorted(p.name for p in OUT.iterdir()))
